### Uncapacitated Facility Location Problem

Classical formulations for the UFLP use two sets of decision variables: one for selecting which facilities to open and another for assigning customer demand to the open facilities. For the location decisions, for each $i \in I$ we define:

$$
y_i =
\begin{cases}
1 & \text{if a facility is open at location } i, \\
0 & \text{otherwise.}
\end{cases}
$$

For the assignment decisions, for each $i \in I$ and $j \in J$ we define:

$$
x_{ij} \geq 0 \text{, demand of customer } j \text{ served by facility } i
$$

A standard integer programming formulation for the UFLP is as follows:

$$
\begin{aligned}
\text{min} \quad & \sum_{i \in I} f_i y_i + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij} \\
\text{subject to} \quad & \sum_{i \in I} x_{ij} = d_j \quad \forall j \in J \\
& \sum_{j \in J} x_{ij} \leq M y_i \quad \forall i \in I \\
& y_i \in \{0,1\} \quad \forall i \in I \\
& x_{ij} \in \{0,1\} \quad \forall i \in I,\; \forall j \in J
\end{aligned}
$$

where $M$ is a sufficiently large constant.

In [1]:
%pip install gurobipy
from gurobipy import Model, GRB, quicksum
import numpy as np

In [2]:
# Number of facilities
facilities = 5

# Number of customers
customers = 5

# Sets
I = range(facilities)
J = range(customers)

# Symmetric shipping cost matrix
c = np.array([[0, 3, 3, 6, 3],
              [3, 0, 4, 5, 5],
              [3, 4, 0, 2, 4],
              [6, 5, 2, 0, 5],
              [3, 5, 4, 5, 0]])

# Facility opening cost
f = np.array([25.0, 25.0, 25.0, 25.0, 25.0])

# Demand
d = np.array([10, 8, 5, 8, 12])

# Big-M
M = np.sum(d)
# M = 1000000000000000000000000000000000

In [6]:
# Define model
m = Model("fixed-charge")

# Decision variables
x = m.addVars(I, J, vtype=GRB.CONTINUOUS, name="x")
y = m.addVars(I, vtype=GRB.BINARY, name="y")

# Objetive function
m.setObjective(quicksum(f[i]*y[i] for i in I) + quicksum(c[i,j]*x[i,j]*d[j] for i in I for j in J), GRB.MINIMIZE)

# Constraints
m.addConstrs(quicksum(x[i,j] for i in I) == 1 for j in J)
m.addConstrs(quicksum(x[i,j]*d[j] for j in J) <= y[i]*M for i in I)

# Optimize
m.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 10 rows, 30 columns and 55 nonzeros (Min)
Model fingerprint: 0x670f0fc6
Model has 25 linear objective coefficients
Variable types: 25 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+01]
  Objective range  [1e+01, 6e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve time: 0.00s
Presolved: 10 rows, 30 columns, 55 nonzeros
Variable types: 25 continuous, 5 integer (5 binary)
Found heuristic solution: objective 125.0000000

Root relaxation: objective 2.500000e+01, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     

In [4]:
if m.status == GRB.OPTIMAL:
    print("Optimal solution")
    print("Total cost: ", m.objVal)
    for i in I:
      if y[i].X > 0.1:
          print("Open facility: ", i)
else:
    print("There is not optimal solution")

Optimal solution
Total cost:  109.0
Open facility:  0
Open facility:  3
Open facility:  4
